In [1]:
from datasets.batch_integration import BatchIntDataset
from utils.experiment_utils import get_all_experiments_info, load_best_model
from utils.eval_utils import compute_mmd_distance, compute_sw_distance
import os

import scvi
import scanpy as sc
import pandas as pd
from collections import defaultdict
from sklearn.neighbors import NearestNeighbors

import hydra
from omegaconf import OmegaConf

import torch
import numpy as np
from geomloss import SamplesLoss

import harmonypy as hm

# silence all warnings
import warnings
warnings.filterwarnings("ignore")

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
n_pcs = 10
train_set = BatchIntDataset(
    root="data",
    split="train",
    n_pcs=n_pcs
)

test_set = BatchIntDataset(
    root="data",
    split="test",
    n_pcs=n_pcs
)

train_donors = train_set.donors
test_donors = test_set.donors
print(train_donors)
print(test_donors)

data loaded !
n train donors: 53
n test donors: 3
data loaded !
n train donors: 53
n test donors: 3
['mouse_pancreatic_islet_atlas_Hrovatin__Fltp_P16__145_mGFP', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_P16__146_mRFP', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_P16__147_mTmG', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse1', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse2', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse3', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse4', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD__SRR10985097', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD__SRR10985098', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD__SRR10985099', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610295', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610296', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610297', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610298', 'mouse_pancreat

In [3]:
configs = get_all_experiments_info('/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/', False)

cfgs = [c for c in configs if 'batchint' in c['name']]

energy_models = [c for c in cfgs if 'mmd' in c['config']['generator'].values()
                 and 'onehot' not in c['name'] and 'GNN' in c['config']['encoder']['_target_']]
                #  and (c['config']['training']['num_epochs'] == 5000 or 'onehot' in c['name'])]
swd_models = [c for c in cfgs if 'swd' in c['config']['generator'].values()
              and 'onehot' not in c['name'] and 'GNN' in c['config']['encoder']['_target_']]
            #   and (c['config']['training']['num_epochs'] == 5000 or 'onehot' in c['name'])]
fm_models = [c for c in cfgs if 'Flow' in c['generator']
             and 'onehot' not in c['name'] and 'GNN' in c['config']['encoder']['_target_']]
            #  and (c['config']['training']['num_epochs'] == 5000 or 'onehot' in c['name'])]

print("energy models:")
for model in energy_models:
    print(model['name'])

print("fm models:")
for model in fm_models:
    print(model['name'])

print("swd models:")
for model in swd_models:
    print(model['name'])

energy models:
batchint_51fd5cc1e1cb76538d5e871f133014cc
fm models:
batchint_fm_dab5345122fa4556bd624bc4fcafa3dd
swd models:
batchint_54db9e4d1d2ac64c78877f1e06a00d6f


In [4]:
def load_model(cfg, path, device):
    enc = hydra.utils.instantiate(cfg['encoder'])
    gen = hydra.utils.instantiate(cfg['generator'])
    state = load_best_model(path)
    enc.load_state_dict(state['encoder_state_dict'])
    gen.load_state_dict(state['generator_state_dict'])
    enc.eval()
    gen.eval()
    enc.to(device)
    gen.to(device)
    return enc, gen

In [10]:
results = {
    'generator' : [],
    'd_pair' : [],
    'd_rand' : []
}

device = 'cuda'
n_samples = 10

for model, name in zip(energy_models+fm_models+swd_models, ['energy']*len(energy_models) + ['flow']*len(fm_models) + ['swd']*len(swd_models)):
    print(f"Evaluating model: {name}")
    enc, gen = load_model(model['config'], model['dir'], device)
    
    for p in range(len(test_set)):
        for _ in range(n_samples):
            batch = test_set[p]

            source_samples = batch['source_samples'].to(device).unsqueeze(0)
            target_samples = batch['target_samples'].to(device).unsqueeze(0)
            
            with torch.no_grad():
                source_latent = enc(source_samples)
                target_latent = enc(target_samples)
                samples = gen.sample(source_samples.reshape(-1, n_pcs), source_latent, target_latent)

            d_pair = (samples - source_samples).norm(dim=1).mean().item()
            results['generator'].append(name)
            results['d_pair'].append(d_pair)
            
            # shuffle source and target samples to get random pairs
            idx = torch.randperm(source_samples.shape[0])
            source_samples_shuffled = source_samples[idx]
            d_rand = (target_samples - source_samples_shuffled).norm(dim=1).mean().item()
            results['d_rand'].append(d_rand)

Evaluating model: energy
Evaluating model: flow
Evaluating model: swd


In [11]:
results_df = pd.DataFrame(results)
# results_df.groupby('generator').mean()
results_df.groupby('generator').mean()['d_pair'] / results_df.groupby('generator').mean()['d_rand']

generator
energy    0.908348
flow      0.429816
swd       0.872895
dtype: float64

In [7]:
cell_type_results = {
    "generator": [],
    "source type" : [],
    "target type" : [],
}

# construct nearest neighbor classifier for cell types

cell_type_classifier = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
cell_type_classifier.fit(test_set.adata.obsm['X_pca'][:, :n_pcs])
cell_types = test_set.adata.obs['cell_type_reannotatedIntegrated'].values

for model, name in zip(energy_models+fm_models+swd_models, ['energy', 'fm', 'swd']):
    print(f"Evaluating model: {model['name']}")
    enc, gen = load_model(model['config'], model['dir'], device)
    
    for p in range(len(test_set)):
        for _ in range(n_samples):
            batch = test_set[p]

            source_samples = batch['source_samples'].to(device).unsqueeze(0)
            target_samples = batch['target_samples'].to(device).unsqueeze(0)
            
            with torch.no_grad():
                source_latent = enc(source_samples)
                target_latent = enc(target_samples)
                samples = gen.sample(source_samples.reshape(-1, n_pcs), source_latent, target_latent)

            # for each sample, use nearest neighbors to find the closest target sample and get its cell type
            source_samples_np = source_samples.cpu().numpy().reshape(-1, n_pcs)
            
            samples_np = samples.cpu().numpy().reshape(-1, n_pcs)

            _, indices = cell_type_classifier.kneighbors(samples_np)
            predicted_cell_types = cell_types[indices.flatten()]

            _, indices = cell_type_classifier.kneighbors(source_samples_np)
            source_cell_types = cell_types[indices.flatten()]

            for s, t in zip(source_cell_types, predicted_cell_types):
                cell_type_results['generator'].append(name)
                cell_type_results['source type'].append(s)
                cell_type_results['target type'].append(t)



Evaluating model: batchint_51fd5cc1e1cb76538d5e871f133014cc
Evaluating model: batchint_fm_dab5345122fa4556bd624bc4fcafa3dd


KeyboardInterrupt: 

In [ ]:
cell_type_results_df = pd.DataFrame(cell_type_results)

grouped = cell_type_results_df.groupby(['generator', 'source type', 'target type']).size()

for generator in grouped.index.levels[0]:
    gen_group = grouped[generator]
    source_types = gen_group.index.levels[0]
    target_types = gen_group.index.levels[1]
    
    transition_matrix = np.zeros((len(source_types), len(target_types)))
    for (s, t), count in gen_group.items():
        transition_matrix[source_types.get_loc(s), target_types.get_loc(t)] = count
    
    total = transition_matrix.sum()
    
    # 1) MSE between true (source) and predicted (target) marginal distributions
    true_dist = transition_matrix.sum(axis=1) / total   # row marginals
    pred_dist = transition_matrix.sum(axis=0) / total   # col marginals
    mse = ((true_dist - pred_dist) ** 2).mean()
    
    # 2) Fraction of off-diagonal mass (only meaningful for shared cell types)
    common = source_types.intersection(target_types)
    common_matrix = np.zeros((len(common), len(common)))
    for i, ct in enumerate(common):
        for j, ct2 in enumerate(common):
            s_idx = source_types.get_loc(ct)
            t_idx = target_types.get_loc(ct2)
            common_matrix[i, j] = transition_matrix[s_idx, t_idx]
    
    diag_frac = np.trace(common_matrix) / common_matrix.sum() if common_matrix.sum() > 0 else 0
    off_diag_frac = 1 - diag_frac
    
    print(f"{generator}: marginal MSE={mse:.4f}, off-diagonal frac={off_diag_frac:.4f}")

energy: marginal MSE=0.0003, off-diagonal frac=0.3453
fm: marginal MSE=0.0000, off-diagonal frac=0.0984
swd: marginal MSE=0.0004, off-diagonal frac=0.3896
